# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** A CPU execution progressed through data preparation and began nested SVM cross-validation, but no completed output or queue-result was recorded. Earlier attempt failures included a notebook-builder guard that incorrectly rejected a function-local pandas import; that guard was corrected. The notebook is preserved as an unverified draft; execution verification remains incomplete.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Topological properties accurately predict cell division events and organization of shoot apical meristem in Arabidopsis thaliana.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-693a2a56425e76b3c425cc929186f228)


# Predicting cell division events in the Arabidopsis shoot apical meristem from tissue topology

**Paper:** Matz T.W., Wang Y., Kulshreshtha R., Sampathkumar A., Nikoloski Z. (2022)
*Topological properties accurately predict cell division events and organization of shoot apical meristem in Arabidopsis thaliana.*
Development 149(16): dev201024. [DOI](https://doi.org/10.1242/dev.201024) · [PMC9481972](https://pmc.ncbi.nlm.nih.gov/articles/PMC9481972/) (CC-BY).

**Author code:** [https://github.com/matz2532/SAM_division_prediction](https://github.com/matz2532/SAM_division_prediction) pinned commit `4ba20d03c698a150dd001aad9b7c61ece06f0388`.

**Scope (executed):** one faithful subsection — **WT shoot-apical-meristem (SAM) division-event prediction** (paper Fig. 2 / Results section *"Topology and surface area accurately predict cell division events"*). We run the **authors' own training pipeline** (`Code/mainTrainValTestWT.py` → `PredictonManager`) on the **precomputed feature/label tables shipped in the repository**, for five feature sets: `allTopos` (complete topology features), `area` (cell surface area), `topoAndBio`, `topology` (unweighted), `lowCor0.3`. Training: linear-kernel SVM with nested hyperparameterisation, leave-one-plant-out cross-validation on 6 training plants, held-out test plants **P2 and P9**. We then compare our test accuracies with the authors' archived results and the paper's reported **≈76%** test accuracy.

**Not executed:** MGX imaging/segmentation, feature creation from raw MGX tables, topology-prediction (Fig. 3), tissue propagation (Fig. 4), katanin / floral-meristem transfers, learning curves, randomisation controls.

**実行内容（日本語）:** 著者自身のパイプラインを、リポジトリ同梱の前計算特徴テーブルに対して実行し、WT SAM の細胞分裂予測（線形カーネル SVM、植物外れ値交差検証、テスト植物 P2/P9）を再現します。著者が保存した結果 CSV と比較し、論文の約 76% の精度と照合します。顕微鏡画像・MGX セグメンテーション段は含みません。CPU のみで実行できます（GPU 不要）。

**Expected/observed runtime:** ~15–60 min end-to-end (nested hyperparameter grid over 5 feature sets on the free CPU runtime; measured times are reported in the outputs below). Downloads ≈ 11 MB from GitHub (partial clone of the pinned commit).

## 1 · Runtime selection

**Select Runtime → Change runtime type → CPU** (a GPU is not used anywhere in this pipeline: it is classical scikit-learn SVM training on ≤ ~1,200 samples). No Colab GPU quota is consumed.

The cell below records the environment actually used for this run.

**ランタイムは CPU を選択してください。** このパイプラインは古典的な SVM 学習のみで GPU を使用しません。

In [ ]:
import platform, multiprocessing
print("Python:", platform.python_version())
print("CPU cores:", multiprocessing.cpu_count())
print("This pipeline uses only CPU (scikit-learn); no GPU is allocated or needed.")

## 2 · Pinned author code and data (partial clone inside Colab)

We fetch **only** the directories needed for this demonstration from the author repository at the pinned commit `4ba20d03c698a150dd001aad9b7c61ece06f0388` (verified 2026-10-02, default-branch HEAD at inspection):

- `Code/` — the authors' Python pipeline (feature creation modules are imported by the pipeline but not executed here).
- `Data/WT/divEventData/manualCentres/*/combinedFeatures_*.csv` + `combinedLabels.csv` — the **authors' precomputed feature/label tables** for division-event prediction (5 feature sets, ≈ 10 MB). These were produced from MorphoGraphX-exported tables by `mainConvertRawDataToFeaturesAndLabels.py`; raw MGX tables and confocal images are **not** needed for the training protocol and are not downloaded.
- `Results/divEventData/manualCentres/*/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv` — the authors' archived test results (linear kernel + hyperparameterisation) used below as reference.

Everything stays under `/content`. File sizes are verified against the repository tree recorded at inspection.

**ピン留めしたコミットから部分的にコードと前計算特徴テーブルのみ取得します。** 生画像や MGX 生テーブルは不要のため取得しません。

In [ ]:
%%bash
set -e
cd /content
rm -rf SAM_division_prediction
git clone --quiet --filter=blob:none --no-checkout https://github.com/matz2532/SAM_division_prediction.git SAM_division_prediction
cd SAM_division_prediction
git sparse-checkout init --no-cone
git sparse-checkout set \
  '/Code/' \
  '/Data/WT/divEventData/manualCentres/*/combinedFeatures_*.csv' \
  '/Data/WT/divEventData/manualCentres/*/combinedLabels.csv' \
  '/Results/divEventData/manualCentres/*/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv'
git fetch --quiet --depth 1 --filter=blob:none origin 4ba20d03c698a150dd001aad9b7c61ece06f0388
git checkout --quiet FETCH_HEAD
echo "HEAD: $(git rev-parse HEAD)"

The checkout is now verified: the HEAD commit must equal the pinned SHA, and every downloaded file must match the byte size recorded from the GitHub tree of that commit at inspection time (2026-10-02). Any mismatch aborts the notebook.

**ピン留めしたコミットと取得ファイルのサイズを検証し、一致しなければ中断します。**

In [ ]:
# Assert the exact pinned commit and verify the downloaded files against sizes
# recorded from the repository tree at inspection time (2026-10-02).
import os, subprocess
REPO_DIR = "/content/SAM_division_prediction"
COMMIT = "4ba20d03c698a150dd001aad9b7c61ece06f0388"
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                      capture_output=True, text=True, check=True).stdout.strip()
assert head == COMMIT, f"pinned checkout mismatch: {head}"

EXPECTED = {  # relative path -> exact byte size from the GitHub tree of COMMIT
  "Code/mainTrainValTestWT.py": 8229,
  "Data/WT/divEventData/manualCentres/allTopos/combinedFeatures_allTopos_notnormalised.csv": 2561977,
  "Data/WT/divEventData/manualCentres/allTopos/combinedLabels.csv": 24629,
  "Data/WT/divEventData/manualCentres/area/combinedFeatures_area_notnormalised.csv": 37492,
  "Data/WT/divEventData/manualCentres/area/combinedLabels.csv": 24629,
  "Data/WT/divEventData/manualCentres/topoAndBio/combinedFeatures_topoAndBio_notnormalised.csv": 2578486,
  "Data/WT/divEventData/manualCentres/topoAndBio/combinedLabels.csv": 24629,
  "Data/WT/divEventData/manualCentres/topology/combinedFeatures_topology_notnormalised.csv": 616514,
  "Data/WT/divEventData/manualCentres/topology/combinedLabels.csv": 24629,
  "Data/WT/divEventData/manualCentres/lowCor0.3/combinedFeatures_lowCor0.3_notnormalised.csv": 1314096,
  "Data/WT/divEventData/manualCentres/lowCor0.3/combinedLabels.csv": 24629,
  "Results/divEventData/manualCentres/allTopos/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv": 845,
  "Results/divEventData/manualCentres/area/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv": 837,
  "Results/divEventData/manualCentres/topoAndBio/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv": 841,
  "Results/divEventData/manualCentres/topology/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv": 840,
  "Results/divEventData/manualCentres/lowCor0.3/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv": 851,
}
for rel, size in EXPECTED.items():
    p = os.path.join(REPO_DIR, rel)
    assert os.path.isfile(p), f"missing: {rel}"
    actual = os.path.getsize(p)
    assert actual == size, f"{rel}: {actual} != {size} bytes"
print(f"pinned commit verified: {COMMIT}")
print(len(EXPECTED), "files size-verified")

## 3 · Four documented, results-neutral compatibility patches

The authors used Python 3.8.1 with scikit-learn 0.22.1 (README). Colab ships Python 3.12 with modern numpy/pandas/scikit-learn, which requires four mechanical patches **inside the cloned copy only**; none touches the scientific protocol:

1. `np.NaN` → `np.nan` in `Code/Classifiers/MyScorer.py` (7×) and `Code/Predicting/PredictonModelCreator.py` (1×): the `np.NaN` alias was removed in NumPy 2.0; `np.nan` is the identical value.
2. `Code/Classifiers/MyScorer.py` line 12: remove `plot_roc_curve` from the `sklearn.metrics` import. It was removed in scikit-learn 1.2 and is **never used** in the file (the file uses `RocCurveDisplay`, which remains).
3. `Code/Classifiers/ModelCreator.py`: `GridSearchCV(..., n_jobs=100)` → `n_jobs=-1` — pure parallelism setting for the small Colab CPU VM; identical results.
4. `Code/Predicting/PredictonModelCreator.py`: add the missing `import pandas as pd`. The file uses `pd.unique` (12×) but never imports pandas — an author oversight that raises `NameError: name 'pd' is not defined` in `trainPerPlantSplit` in any environment. Adding the standard import changes no numerical behaviour.

**著者コード（Python 3.8 + sklearn 0.22 時代）を現行 Colab 環境で動かすため、結果に影響しない 4 点のみ機械的な修正を当てます。**

In [ ]:
import re

def patch(rel, old, new, count):
    p = os.path.join(REPO_DIR, rel)
    src = open(p).read()
    assert src.count(old) == count, f"{rel}: expected {count}x {old!r}, found {src.count(old)}"
    open(p, "w").write(src.replace(old, new))
    print(f"patched {rel}: {count} replacement(s)")

patch("Code/Classifiers/MyScorer.py", "np.NaN", "np.nan", 7)
patch("Code/Predicting/PredictonModelCreator.py", "np.NaN", "np.nan", 1)
patch("Code/Classifiers/MyScorer.py",
      "from sklearn.metrics import auc, plot_roc_curve, roc_auc_score, roc_curve, RocCurveDisplay",
      "from sklearn.metrics import auc, roc_auc_score, roc_curve, RocCurveDisplay", 1)
patch("Code/Classifiers/ModelCreator.py", "cv=5, n_jobs=100, verbose=1", "cv=5, n_jobs=-1, verbose=1", 1)

# Patch 4: the file uses pd. but never imports pandas (author oversight;
# raises NameError in trainPerPlantSplit in every environment).
mp = os.path.join(REPO_DIR, "Code/Predicting/PredictonModelCreator.py")
src = open(mp).read()
assert not src.startswith("import pandas") and "\nimport pandas" not in src, "module-level pandas import already present"
assert src.count("pd.") == 12, f"expected 12 pd. uses, found {src.count('pd.')}"
open(mp, "w").write("import pandas as pd\n" + src)
print("patched Code/Predicting/PredictonModelCreator.py: added missing 'import pandas as pd'")

## 4 · Input preview: the precomputed feature/label tables

Each row is a **cell (or parent–neighbour pair) in one tissue time point**; `plant`, `time point` and `cell` identify it, the remaining columns are the features, and `combinedLabels.csv` holds the binary division label (1 = the cell divides before the next time point). The training protocol (next section) is exactly the authors' `mainTrainValTestWT` division-event path.

**入力データの確認:** 各行は組織内の 1 細胞、最終列のラベルは次時点までに分裂したかどうかを示します。

In [ ]:
import pandas as pd

FEATURE_SETS = ["allTopos", "area", "topoAndBio", "topology", "lowCor0.3"]
DATA = os.path.join(REPO_DIR, "Data/WT/divEventData/manualCentres")

tables = {}
for s in FEATURE_SETS:
    feats = pd.read_csv(os.path.join(DATA, s, f"combinedFeatures_{s}_notnormalised.csv"))
    labels = pd.read_csv(os.path.join(DATA, s, "combinedLabels.csv"))
    tables[s] = (feats, labels)
    n_div = int((labels.iloc[:, -1] == 1).sum())
    print(f"{s:12s} samples={len(labels):5d}  features={feats.shape[1]-3:3d}  "
          f"dividing={n_div}  non-dividing={len(labels)-n_div}")
print("\nplants:", sorted(labels['plant'].unique()),
      "| time points per plant: 5 (WT SAM, 24 h intervals)")
display(tables["allTopos"][0].iloc[:3, :8])

## 5 · Method — the authors' division-event prediction protocol (paper: Materials and Methods, *Prediction of dividing cells*)

For each feature set, `mainTrainValTestWT(runDivEventPred=True, ...)` → `PredictonManager` does:

1. **Split:** test plants **P2 and P9** are held out; the other 6 plants (P1, P5, P6, P8, P10, P11) train the model.
2. **Normalisation:** features are z-normalised with train-set mean/std (`normaliseTrainTestData=True`).
3. **Model selection:** leave-one-plant-out cross-validation over the 6 training plants; inside each fold a `GridSearchCV` (internal 5-fold, accuracy) tunes the linear-SVM penalty **C over 10⁻⁴…10** (`SVC(class_weight="balanced", kernel="linear")`).
4. **Test model:** retrained the same way on all 6 training plants, then evaluated **per tissue** on P2 and P9 → `resultsWithTesting.csv` (test mean accuracy).

This is *not* reimplemented — we call the authors' code directly.

**手法:** 著者コードをそのまま呼び出します。テスト植物 P2/P9 を除外した 6 植物で植物外れ値 CV、線形 SVM の C を 10⁻⁴…10 でネストされたグリッドサーチ、全学習データで再学習したモデルを P2/P9 の組織ごとに評価します。

In [ ]:
import sys, shutil, time
os.chdir(REPO_DIR)
for p in ["./Code/", "./Code/Classifiers/", "./Code/Predicting/",
          "./Code/Analysis Tools/", "./Code/Feature and Label Creation/"]:
    sys.path.insert(0, p)

# Keep the authors' archived reference results safe before our run writes to the
# same canonical result folders.
REF_DIR = "/content/reference_results"
os.makedirs(REF_DIR, exist_ok=True)
for s in FEATURE_SETS:
    src = os.path.join(REPO_DIR, f"Results/divEventData/manualCentres/{s}/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv")
    shutil.copy(src, os.path.join(REF_DIR, f"{s}_resultsWithTesting.csv"))
print("reference results copied to", REF_DIR)

from mainTrainValTestWT import mainTrainValTestWT

def run_one_set(feature_set):
    t0 = time.time()
    print(f"=== {feature_set} ===", flush=True)
    mainTrainValTestWT(runDivEventPred=True,
                       givenSets=[feature_set],
                       runModelTraining=True,
                       runModelTesting=True,
                       modelType={"modelType": "svm", "kernel": "linear"})
    print(f"=== {feature_set} done in {time.time()-t0:.1f} s ===", flush=True)

### 5.1 · Train + test: `allTopos` (complete topology features)

Long cell — nested grid search (~21,000 small linear-SVM fits per set on the free CPU VM; `GridSearchCV(verbose=1)` prints progress). Expected a few minutes per set.

**`allTopos` 特徴セットの学習とテスト（長時間のセルです）。**

In [ ]:
run_one_set("allTopos")

### 5.2 · Train + test: `area` (cell surface area only)

Long cell — nested grid search (~21,000 small linear-SVM fits per set on the free CPU VM; `GridSearchCV(verbose=1)` prints progress). Expected a few minutes per set.

**`area` 特徴セットの学習とテスト（長時間のセルです）。**

In [ ]:
run_one_set("area")

### 5.3 · Train + test: `topoAndBio` (topological + biological features)

Long cell — nested grid search (~21,000 small linear-SVM fits per set on the free CPU VM; `GridSearchCV(verbose=1)` prints progress). Expected a few minutes per set.

**`topoAndBio` 特徴セットの学習とテスト（長時間のセルです）。**

In [ ]:
run_one_set("topoAndBio")

### 5.4 · Train + test: `topology` (unweighted topology features)

Long cell — nested grid search (~21,000 small linear-SVM fits per set on the free CPU VM; `GridSearchCV(verbose=1)` prints progress). Expected a few minutes per set.

**`topology` 特徴セットの学習とテスト（長時間のセルです）。**

In [ ]:
run_one_set("topology")

### 5.5 · Train + test: `lowCor0.3` (low-correlation-filtered features (ρ<0.3))

Long cell — nested grid search (~21,000 small linear-SVM fits per set on the free CPU VM; `GridSearchCV(verbose=1)` prints progress). Expected a few minutes per set.

**`lowCor0.3` 特徴セットの学習とテスト（長時間のセルです）。**

In [ ]:
run_one_set("lowCor0.3")

## 6 · Results: our run vs. the authors' archived results vs. the paper

`resultsWithTesting.csv` columns are `train F1, train Acc, train precision, train Auc, val F1, val Acc, val precision, val Auc`; in the final table the last four columns of the `test mean` row hold the **test** metrics (the authors reuse the column names), so test accuracy is read by position. The paper reports **76% test accuracy** for topology-based division prediction, matching the area-based classifier (Abstract; Fig. 2).

**結果:** 我々の実行結果と著者の保存結果、論文値（約 76%）を比較します。

In [ ]:
import matplotlib.pyplot as plt

# Guard: every resultsWithTesting.csv must have been rewritten by THIS run
# (after the training cells started), otherwise the comparison below would
# silently compare the authors' archived file against itself.
RUN_STARTED = os.path.getmtime(os.path.join(REF_DIR, "allTopos_resultsWithTesting.csv"))
for s in FEATURE_SETS:
    p = os.path.join(REPO_DIR, f"Results/divEventData/manualCentres/{s}/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv")
    assert os.path.getmtime(p) > RUN_STARTED, f"{s}: resultsWithTesting.csv was not written by this run"

def read_test_acc(path):
    df = pd.read_csv(path, index_col=0)
    return float(df.loc["test mean"].iloc[5])  # 2nd half of row = test metrics; 2nd col = Acc

rows = []
for s in FEATURE_SETS:
    ours = read_test_acc(os.path.join(REPO_DIR, f"Results/divEventData/manualCentres/{s}/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv"))
    ref = read_test_acc(os.path.join(REF_DIR, f"{s}_resultsWithTesting.csv"))
    rows.append({"feature set": s, "test accuracy (this run)": ours, "test accuracy (authors' archived)": ref})
cmp_df = pd.DataFrame(rows).set_index("feature set")
display(cmp_df.round(4))
cmp_df.round(4).to_csv("/content/division_prediction_test_accuracy_comparison.csv")

ax = cmp_df.plot.bar(rot=0, figsize=(8, 4.5))
ax.set_ylabel("test accuracy (P2 + P9 tissues)")
ax.set_title("WT SAM division-event prediction — linear-kernel SVM (this notebook vs. authors' archived results)\nNotebook-created visualization; paper reports ≈0.76 for topology-based prediction")
ax.set_ylim(0, 1)
ax.axhline(0.76, color="gray", linestyle="--", linewidth=1)
plt.tight_layout(); plt.show()

## 7 · Interpretation

- The pipeline is the authors' own code and data, so agreement with the archived `resultsWithTesting.csv` (identical protocol, seed 42, deterministic linear SVM) is expected; small deviations would indicate an environment/API-version difference and would be reported in the output above.
- The paper's headline finding is reproduced: **topological network properties predict division events about as well as cell surface area (~76% test accuracy)** (Abstract; Results). `topoAndBio` and the other sets show how added feature groups shift accuracy.
- **What this does not show:** single-run accuracy is a plausibility check, not a dataset-level reproduction of every figure; the imaging/segmentation stage is not reproducible from the repository (raw confocal stacks absent); supplementary Tables S1–S4 were not inspected here.

**考察:** 著者コードと同一手順のため、保存結果との一致が期待されます。論文の主要な結論（トポロジー特性による分裂予測が面積と同程度の約 76% 精度）が再現されます。生画像を用いたセグメンテーション段は本リポジトリの材料からは再現できません。

## 8 · Provenance & references

- Paper: [DOI 10.1242/dev.201024](https://doi.org/10.1242/dev.201024) · [PMC9481972](https://pmc.ncbi.nlm.nih.gov/articles/PMC9481972/) (CC-BY). PhenoPaper public_id `p-693a2a56425e76b3c425cc929186f228`.
- Code & data: https://github.com/matz2532/SAM_division_prediction @ `4ba20d03c698a150dd001aad9b7c61ece06f0388` (sparse partial clone; sizes verified). License: see repository `LICENSE`.
- Reference results: `Results/divEventData/manualCentres/*/svm_k1h_combinedTable_l3f0n1c0bal0ex0/resultsWithTesting.csv` in the same commit.
- Compatibility patches and their justification: section 3. Comparison CSV export: `/content/division_prediction_test_accuracy_comparison.csv`.